# Environment and Model Baseline

Establish the exact software, cluster, model, tokenizer, and benchmark configuration that later inference results depend on.

## Objectives

- Identify the local and peer nodes and record relevant software versions.
- Record visible CUDA devices and verify both direct network rails without changing them.
- Define model and tokenizer metadata without automatically downloading artifacts.
- Define benchmark metrics and controlled workload dimensions for later notebooks.

## Background

Inference results are comparable only when software, cluster, model, tokenizer, and workload definitions are recorded consistently. Optional components are reported as facts, not assumed to exist.

## Prediction

The two DGX Spark nodes should form a reproducible execution baseline before any inference workload is launched.

Specifically:

- the local node should identify itself as `spark-0240`;
- the peer should identify itself as `spark-f868`;
- both nodes should report the `aarch64` machine architecture;
- both repository checkouts should resolve to the same Git commit;
- both checkouts should be clean, or any local modifications should be reported explicitly;
- non-interactive SSH to the peer should complete without requiring user input.

A matching repository revision and architecture will establish configuration parity only for the facts measured here. It will not establish parity of Python packages, CUDA, drivers, model artifacts, containers, network state, or runtime behavior.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Repository root: {repository_root}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Repository root: /home/coert/workspace/dgx-spark-lab
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference


## Experiment

Complete configuration placeholders before running active measurement cells.

### Optional package detection

In [2]:
from importlib.metadata import PackageNotFoundError, version

packages = (
    "torch",
    "vllm",
    "transformers",
    "tokenizers",
    "huggingface-hub",
    "ray",
    "pandas",
)
package_versions = {}
for package in packages:
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = "not installed"
package_versions

{'torch': '2.13.0',
 'vllm': 'not installed',
 'transformers': 'not installed',
 'tokenizers': 'not installed',
 'huggingface-hub': 'not installed',
 'ray': 'not installed',
 'pandas': '3.0.5'}

### CUDA detection

This check reports device facts and does not allocate large tensors.

In [3]:
from dataclasses import asdict
from common.cuda import detect_cuda

cuda_facts = asdict(detect_cuda())
cuda_facts

{'torch_installed': True,
 'available': True,
 'device_count': 1,
 'device_names': ('NVIDIA GB10',),
 'torch_version': '2.13.0+cu130',
 'cuda_version': '13.0',
 'error': None}

### Cluster configuration

Read the repository configuration without requiring a dotenv package. This cell does not contact the peer.

In [4]:
def read_env_file(path: Path) -> dict[str, str]:
    values = {}
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator:
            raise ValueError(f"Invalid configuration line: {raw_line!r}")
        values[key.strip()] = value.strip()
    return values


cluster = read_env_file(repository_root / "config" / "cluster.env")
cluster_facts = {
    key: cluster.get(key)
    for key in (
        "HEAD_HOST",
        "HEAD_IP",
        "WORKER_HOST",
        "WORKER_IP",
        "ETH_IF_0",
        "IB_IF_0",
        "SUBNET_0",
        "ETH_IF_1",
        "IB_IF_1",
        "SUBNET_1",
    )
}
cluster_facts

{'HEAD_HOST': 'spark-0240',
 'HEAD_IP': '10.200.0.1',
 'WORKER_HOST': 'spark-f868',
 'WORKER_IP': '10.200.0.2',
 'ETH_IF_0': 'enP2p1s0f1np1',
 'IB_IF_0': 'roceP2p1s0f1',
 'SUBNET_0': '10.200.0.0/30',
 'ETH_IF_1': 'enp1s0f1np1',
 'IB_IF_1': 'rocep1s0f1',
 'SUBNET_1': '10.201.0.0/30'}

### Repository and node identity

Verify the local and peer hostnames, machine architectures, repository revisions, and working-tree states.

The peer checks use non-interactive SSH and do not modify either system. A failed command remains part of the result rather than being silently discarded.

In [9]:
import shlex
import subprocess
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True, slots=True)
class CommandResult:
    node: str
    check: str
    command: str
    returncode: int
    stdout: str
    stderr: str

    @property
    def succeeded(self) -> bool:
        return self.returncode == 0


def run_command(
    *,
    node: str,
    check: str,
    command: list[str],
    timeout_s: float = 15.0,
) -> CommandResult:
    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_s,
            check=False,
        )
        return CommandResult(
            node=node,
            check=check,
            command=shlex.join(command),
            returncode=completed.returncode,
            stdout=completed.stdout.strip(),
            stderr=completed.stderr.strip(),
        )
    except subprocess.TimeoutExpired as exc:
        return CommandResult(
            node=node,
            check=check,
            command=shlex.join(command),
            returncode=124,
            stdout=(exc.stdout or "").strip(),
            stderr=f"Timed out after {timeout_s:.1f} seconds",
        )
    except OSError as exc:
        return CommandResult(
            node=node,
            check=check,
            command=shlex.join(command),
            returncode=127,
            stdout="",
            stderr=str(exc),
        )


HEAD_HOST = cluster["HEAD_HOST"]
WORKER_HOST = cluster["WORKER_HOST"]

REMOTE_REPOSITORY_ROOT = str(repository_root)

local_checks = {
    "hostname": ["hostname"],
    "machine_architecture": ["uname", "-m"],
    "git_revision": [
        "git",
        "-C",
        str(repository_root),
        "rev-parse",
        "HEAD",
    ],
    "git_branch": [
        "git",
        "-C",
        str(repository_root),
        "branch",
        "--show-current",
    ],
    "git_status": [
        "git",
        "-C",
        str(repository_root),
        "status",
        "--porcelain",
        "--untracked-files=normal",
    ],
}

remote_shell_checks = {
    "hostname": "hostname",
    "machine_architecture": "uname -m",
    "git_revision": (f"git -C {shlex.quote(REMOTE_REPOSITORY_ROOT)} rev-parse HEAD"),
    "git_branch": (
        f"git -C {shlex.quote(REMOTE_REPOSITORY_ROOT)} branch --show-current"
    ),
    "git_status": (
        f"git -C {shlex.quote(REMOTE_REPOSITORY_ROOT)} "
        "status --porcelain --untracked-files=normal"
    ),
}

identity_results: list[CommandResult] = []

for check, command in local_checks.items():
    identity_results.append(
        run_command(
            node=HEAD_HOST,
            check=check,
            command=command,
        )
    )

for check, remote_command in remote_shell_checks.items():
    identity_results.append(
        run_command(
            node=WORKER_HOST,
            check=check,
            command=[
                "ssh",
                "-o",
                "BatchMode=yes",
                "-o",
                "ConnectTimeout=5",
                WORKER_HOST,
                remote_command,
            ],
        )
    )

identity_results_df = pd.DataFrame(
    [
        {
            **asdict(result),
            "succeeded": result.succeeded,
        }
        for result in identity_results
    ]
)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(identity_results_df)

,node,check,command,returncode,stdout,stderr,succeeded
0,spark-0240,hostname,hostname,0,spark-0240,,True
1,spark-0240,machine_architecture,uname -m,0,aarch64,,True
2,spark-0240,git_revision,git -C /home/coert/workspace/dgx-spark-lab rev...,0,206fb575f5cd8713a2037617750a1df6a0240ba6,,True
3,spark-0240,git_branch,git -C /home/coert/workspace/dgx-spark-lab bra...,0,main,,True
4,spark-0240,git_status,git -C /home/coert/workspace/dgx-spark-lab sta...,0,,,True
5,spark-f868,hostname,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,spark-f868,,True
6,spark-f868,machine_architecture,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,aarch64,,True
7,spark-f868,git_revision,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,206fb575f5cd8713a2037617750a1df6a0240ba6,,True
8,spark-f868,git_branch,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,main,,True
9,spark-f868,git_status,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,,,True


In [10]:
def result_value(node: str, check: str) -> str | None:
    matches = identity_results_df[
        (identity_results_df["node"] == node) & (identity_results_df["check"] == check)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one result for node={node!r}, check={check!r}; "
            f"found {len(matches)}"
        )

    row = matches.iloc[0]
    return row["stdout"] if row["succeeded"] else None


local_hostname = result_value(HEAD_HOST, "hostname")
peer_hostname = result_value(WORKER_HOST, "hostname")

local_architecture = result_value(HEAD_HOST, "machine_architecture")
peer_architecture = result_value(WORKER_HOST, "machine_architecture")

local_revision = result_value(HEAD_HOST, "git_revision")
peer_revision = result_value(WORKER_HOST, "git_revision")

local_branch = result_value(HEAD_HOST, "git_branch")
peer_branch = result_value(WORKER_HOST, "git_branch")

local_status = result_value(HEAD_HOST, "git_status")
peer_status = result_value(WORKER_HOST, "git_status")

identity_summary = pd.Series(
    {
        "all_commands_succeeded": bool(identity_results_df["succeeded"].all()),
        "local_hostname": local_hostname,
        "peer_hostname": peer_hostname,
        "hostnames_match_configuration": (
            local_hostname == HEAD_HOST and peer_hostname == WORKER_HOST
        ),
        "local_architecture": local_architecture,
        "peer_architecture": peer_architecture,
        "both_nodes_are_aarch64": (
            local_architecture == "aarch64" and peer_architecture == "aarch64"
        ),
        "local_git_revision": local_revision,
        "peer_git_revision": peer_revision,
        "git_revisions_match": (
            local_revision is not None and local_revision == peer_revision
        ),
        "local_branch": local_branch,
        "peer_branch": peer_branch,
        "both_branches_are_main": (local_branch == "main" and peer_branch == "main"),
        "local_worktree_clean": local_status == "",
        "peer_worktree_clean": peer_status == "",
    },
    name="value",
)

identity_summary.to_frame()

,value
all_commands_succeeded,True
local_hostname,spark-0240
peer_hostname,spark-f868
hostnames_match_configuration,True
local_architecture,aarch64
peer_architecture,aarch64
both_nodes_are_aarch64,True
local_git_revision,206fb575f5cd8713a2037617750a1df6a0240ba6
peer_git_revision,206fb575f5cd8713a2037617750a1df6a0240ba6
git_revisions_match,True


### Peer repository discovery

The initial peer Git checks assumed that both nodes used the same absolute repository path. That assumption was false: SSH and basic peer commands succeeded, but Git could not enter the local node's repository path on the peer.

Search a bounded portion of the peer user's home directory for Git working trees named `dgx-spark-lab`. This is a read-only discovery step and does not update the peer checkout.

In [11]:
peer_repository_discovery = run_command(
    node=WORKER_HOST,
    check="repository_discovery",
    command=[
        "ssh",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=5",
        WORKER_HOST,
        (
            'find "$HOME" '
            "-maxdepth 5 "
            "-type d "
            "-name dgx-spark-lab "
            "-exec test -d '{}/.git' ';' "
            "-print"
        ),
    ],
    timeout_s=30.0,
)

pd.Series(
    {
        "node": peer_repository_discovery.node,
        "returncode": peer_repository_discovery.returncode,
        "succeeded": peer_repository_discovery.succeeded,
        "repository_candidates": (
            peer_repository_discovery.stdout.splitlines()
            if peer_repository_discovery.stdout
            else []
        ),
        "stderr": peer_repository_discovery.stderr,
    },
    name="value",
).to_frame()

,value
node,spark-f868
returncode,0
succeeded,True
repository_candidates,[/home/coert/workspace/dgx-spark-lab]
stderr,


In [12]:
peer_repository_candidates = [
    Path(candidate)
    for candidate in peer_repository_discovery.stdout.splitlines()
    if candidate.strip()
]

if len(peer_repository_candidates) == 1:
    peer_repository_root = peer_repository_candidates[0]
    print(f"Peer repository root: {peer_repository_root}")
elif not peer_repository_candidates:
    peer_repository_root = None
    print("No peer dgx-spark-lab Git checkout found")
else:
    peer_repository_root = None
    print("Multiple peer dgx-spark-lab checkouts found:")
    for candidate in peer_repository_candidates:
        print(f"- {candidate}")

Peer repository root: /home/coert/workspace/dgx-spark-lab


### Host runtime parity

Matching Git revisions do not guarantee matching execution environments.

Record the operating-system kernel, default Python interpreter, `uv`, NVIDIA driver, visible GPU, Docker installation, and presence of the configured vLLM container image on both nodes.

These checks are read-only. Package parity inside a vLLM container will be measured separately because the notebook's Python environment is not necessarily the inference-serving environment.

In [ ]:
runtime_shell_checks = {
    "kernel": "uname -r",
    "operating_system": ('. /etc/os-release && printf "%s %s" "$NAME" "$VERSION_ID"'),
    "python_command": ("command -v python3 || command -v python || true"),
    "python_version": (
        "if command -v python3 >/dev/null 2>&1; then "
        "python3 --version; "
        "elif command -v python >/dev/null 2>&1; then "
        "python --version; "
        "else "
        'printf "not found"; '
        "fi"
    ),
    "uv_version": (
        "if command -v uv >/dev/null 2>&1; then "
        "uv --version; "
        "else "
        'printf "not found"; '
        "fi"
    ),
    "nvidia_driver": (
        "if command -v nvidia-smi >/dev/null 2>&1; then "
        "nvidia-smi "
        "--query-gpu=driver_version "
        "--format=csv,noheader; "
        "else "
        'printf "nvidia-smi not found"; '
        "fi"
    ),
    "gpu_name": (
        "if command -v nvidia-smi >/dev/null 2>&1; then "
        "nvidia-smi "
        "--query-gpu=name "
        "--format=csv,noheader; "
        "else "
        'printf "nvidia-smi not found"; '
        "fi"
    ),
    "docker_version": (
        "if command -v docker >/dev/null 2>&1; then "
        "docker version --format '{{.Client.Version}}'; "
        "else "
        'printf "not found"; '
        "fi"
    ),
    "vllm_image": (
        "if command -v docker >/dev/null 2>&1; then "
        f"docker image inspect {shlex.quote(cluster['VLLM_IMAGE'])} "
        "--format '{{.Id}}' 2>/dev/null "
        '|| printf "not present"; '
        "else "
        'printf "docker not found"; '
        "fi"
    ),
}


def run_shell_check(
    *,
    node: str,
    check: str,
    shell_command: str,
    remote: bool,
) -> CommandResult:
    if remote:
        command = [
            "ssh",
            "-o",
            "BatchMode=yes",
            "-o",
            "ConnectTimeout=5",
            node,
            "bash",
            "-lc",
            shlex.quote(shell_command),
        ]
    else:
        command = [
            "bash",
            "-lc",
            shell_command,
        ]

    return run_command(
        node=node,
        check=check,
        command=command,
        timeout_s=20.0,
    )


runtime_results: list[CommandResult] = []

for check, shell_command in runtime_shell_checks.items():
    runtime_results.append(
        run_shell_check(
            node=HEAD_HOST,
            check=check,
            shell_command=shell_command,
            remote=False,
        )
    )
    runtime_results.append(
        run_shell_check(
            node=WORKER_HOST,
            check=check,
            shell_command=shell_command,
            remote=True,
        )
    )

runtime_results_df = pd.DataFrame(
    [
        {
            **asdict(result),
            "succeeded": result.succeeded,
        }
        for result in runtime_results
    ]
)

runtime_results_df

,node,check,command,returncode,stdout,stderr,succeeded
0,spark-0240,kernel,bash -lc 'uname -r',0,6.17.0-1026-nvidia,,True
1,spark-f868,kernel,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,6.17.0-1026-nvidia,,True
2,spark-0240,operating_system,"bash -lc '. /etc/os-release && printf ""%s %s"" ...",0,Ubuntu 24.04,,True
3,spark-f868,operating_system,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,Ubuntu 24.04,,True
4,spark-0240,python_command,bash -lc 'command -v python3 || command -v pyt...,0,/usr/bin/python3,,True
5,spark-f868,python_command,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,/usr/bin/python3,,True
6,spark-0240,python_version,bash -lc 'if command -v python3 >/dev/null 2>&...,0,Python 3.12.3,,True
7,spark-f868,python_version,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,Python 3.12.3,,True
8,spark-0240,uv_version,bash -lc 'if command -v uv >/dev/null 2>&1; th...,0,uv 0.10.6,,True
9,spark-f868,uv_version,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,uv 0.11.32 (aarch64-unknown-linux-gnu),,True


In [ ]:
runtime_comparison = runtime_results_df.pivot(
    index="check",
    columns="node",
    values="stdout",
).rename_axis(columns=None)

runtime_comparison["matches"] = (
    runtime_comparison[HEAD_HOST] == runtime_comparison[WORKER_HOST]
)

runtime_comparison

,spark-0240,spark-f868,matches
check,,,
docker_version,29.2.1,29.2.1,True
gpu_name,NVIDIA GB10,NVIDIA GB10,True
kernel,6.17.0-1026-nvidia,6.17.0-1026-nvidia,True
nvidia_driver,580.159.03,580.159.03,True
operating_system,Ubuntu 24.04,Ubuntu 24.04,True
python_command,/usr/bin/python3,/usr/bin/python3,True
python_version,Python 3.12.3,Python 3.12.3,True
uv_version,uv 0.10.6,uv 0.11.32 (aarch64-unknown-linux-gnu),False
vllm_image,sha256:3a4454e4b771db79490556b9c278e7b4c20128e...,sha256:3a4454e4b771db79490556b9c278e7b4c20128e...,True


In [ ]:
runtime_failures = runtime_results_df[~runtime_results_df["succeeded"]][
    ["node", "check", "returncode", "stderr"]
]

runtime_summary = pd.Series(
    {
        "all_runtime_commands_succeeded": bool(runtime_results_df["succeeded"].all()),
        "checks_compared": len(runtime_comparison),
        "matching_checks": int(runtime_comparison["matches"].sum()),
        "mismatching_checks": int((~runtime_comparison["matches"]).sum()),
        "vllm_image_matches": bool(runtime_comparison.loc["vllm_image", "matches"]),
    },
    name="value",
)

display(runtime_summary.to_frame())

if not runtime_failures.empty:
    display(runtime_failures)

,value
all_runtime_commands_succeeded,True
checks_compared,9
matching_checks,8
mismatching_checks,1
vllm_image_matches,True


### Model configuration

Choose either a registry identifier or a local path. No model is selected or downloaded by default.

In [ ]:
MODEL_ID = None
TOKENIZER_ID = None
MODEL_REVISION = None
LOCAL_MODEL_PATH = None
TRUST_REMOTE_CODE = False

if MODEL_ID and LOCAL_MODEL_PATH:
    raise ValueError("Set MODEL_ID or LOCAL_MODEL_PATH, not both")
if LOCAL_MODEL_PATH is not None:
    local_model_path = Path(LOCAL_MODEL_PATH).expanduser()
    if not local_model_path.is_dir():
        raise FileNotFoundError(local_model_path)

model_configuration = {
    "model_id": MODEL_ID,
    "tokenizer_id": TOKENIZER_ID,
    "revision": MODEL_REVISION,
    "local_path": str(LOCAL_MODEL_PATH) if LOCAL_MODEL_PATH else None,
    "trust_remote_code": TRUST_REMOTE_CODE,
}
model_configuration

### Benchmark metric definitions

In [ ]:
import pandas as pd

metric_definitions = pd.DataFrame(
    [
        (
            "server_startup_time_s",
            "monotonic duration",
            "Server process start to readiness",
        ),
        (
            "model_load_time_s",
            "server-reported or instrumented duration",
            "Model load boundary must be stated",
        ),
        ("ttft_s", "monotonic duration", "Request start to first streamed token"),
        (
            "inter_token_latency_s",
            "monotonic duration",
            "Interval between successive streamed tokens",
        ),
        (
            "prompt_tokens_per_s",
            "derived rate",
            "Processed prompt tokens per defined prefill interval",
        ),
        (
            "generation_tokens_per_s",
            "derived rate",
            "Generated tokens per defined decode interval",
        ),
        ("end_to_end_latency_s", "monotonic duration", "Request start to completion"),
        (
            "requests_per_s",
            "derived rate",
            "Completed requests per measurement interval",
        ),
        (
            "allocator_memory_bytes",
            "peak or steady measured value",
            "Allocator scope and sampling boundary required",
        ),
        (
            "output_correct",
            "validation result",
            "Result of an explicitly defined correctness check",
        ),
        ("failure_count", "count", "Failed trials retained in the raw data"),
    ],
    columns=("metric", "kind", "definition"),
)
metric_definitions

### Workload dimensions

In [ ]:
PROMPT_TOKEN_COUNTS = []
GENERATED_TOKEN_COUNTS = []
BATCH_SIZES = []
CONCURRENCY_LEVELS = []
WARMUP_COUNT = None
MEASURED_REPETITIONS = None
RANDOM_SEED = None
SAMPLING_SETTINGS = {"temperature": None, "top_p": None}

workload_dimensions = {
    "prompt_token_counts": PROMPT_TOKEN_COUNTS,
    "generated_token_counts": GENERATED_TOKEN_COUNTS,
    "batch_sizes": BATCH_SIZES,
    "concurrency_levels": CONCURRENCY_LEVELS,
    "warmup_count": WARMUP_COUNT,
    "measured_repetitions": MEASURED_REPETITIONS,
    "random_seed": RANDOM_SEED,
    "sampling": SAMPLING_SETTINGS,
}
workload_dimensions

## Observations

TODO: Record only facts produced by the saved outputs of this notebook.

## Explanation

TODO: Explain the measured results. Separate derived values and architectural inference from direct observations.

## Connection to LLMs

TODO: Connect the verified result to inference behavior without claiming effects that were not measured.

## Further Exploration

TODO: Identify the next controlled experiment justified by the result.